# Pseudobulk preprocessing

**Environment:** `clamp-analyses`

Preprocesses a single pseudobulk dataset from raw counts to a z-scored expression matrix and estimates model rank via Gavish-Donoho. Writes `norm.csv` and `k.csv` to `output/01_model_building/05_pseudobulk/<DATASET>/`. These files are consumed by all downstream model notebooks.

## Libraries

In [17]:
library(data.table)
library(rsvd)
library(here)
library(CLAMP)
library(PCAtools)

set.seed(123)

## Parameters

In [18]:
DATASET     = "PBMC_Perez2022"
MEAN_CUTOFF = 0.5
VAR_CUTOFF  = 0.1
OUT_ROOT    = "output/01_model_building/05_pseudobulk"
DATA_DIR    = "data/pseudobulk"

## Preprocess and estimate rank

In [19]:
out_dir <- file.path(here(), OUT_ROOT, DATASET)
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# Load raw counts
raw    <- fread(file.path(here(), DATA_DIR, DATASET, "bulk_expr.csv"))
genes  <- raw[[1]]; raw[[1]] <- NULL
counts <- as.matrix(raw); storage.mode(counts) <- "numeric"
rownames(counts) <- genes
cat("Raw counts:", nrow(counts), "genes x", ncol(counts), "samples\n")

# Preprocess: CPM -> filter -> z-score
cpm  <- CLAMP::cpmCLAMP(counts)
prep <- CLAMP::preprocessCLAMP(Y = cpm, mean_cutoff = MEAN_CUTOFF, var_cutoff = VAR_CUTOFF)
norm <- CLAMP::zscoreCLAMP(Y_filtered = prep$Y_filtered, rowStats = prep$rowStats)
cat("Preprocessed:", nrow(norm), "genes x", ncol(norm), "samples\n")

# SVD
g_fb       <- nrow(norm)
samples_fb <- ncol(norm)
SVD_K      <- floor((min(g_fb, samples_fb) - 1) / 4)
message("SVD K = ", SVD_K)
svdres <- rsvd(norm, k = SVD_K)

# Estimate model rank (Gavish-Donoho x2)
n_genes   <- nrow(norm)
n_samples <- ncol(norm)

k <- num.pc(
  data = norm,
  method = "elbow",
) * 2

k <- max(as.integer(k), 2L)
k <- min(k, SVD_K)  # cannot use more singular vectors than computed
message("Inferred k = ", k)

# Save outputs
write.csv(as.data.frame(norm), file.path(out_dir, "norm.csv"))
write.csv(data.frame(k = k),  file.path(out_dir, "k.csv"), row.names = FALSE)
cat("Saved norm.csv and k.csv to", out_dir, "\n")

Warning message in fread(file.path(here(), DATA_DIR, DATASET, "bulk_expr.csv")):
“Detected 209 column names but the data has 210 columns (i.e. invalid file). Added an extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”


Raw counts: 18854 genes x 209 samples
Preprocessed: 13221 genes x 209 samples


SVD K = 52

Computing SVD

Inferred k = 152



Saved norm.csv and k.csv to /home/msubirana/Documents/pivlab/clamp-analyses/output/01_model_building/05_pseudobulk/PBMC_Perez2022 
